# ==========================================================
# TWITTER SENTIMENT ANALYSIS USING MACHINE LEARNING
# ==========================================================

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from wordcloud import WordCloud


In [ ]:
# Download NLTK Data

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# LOAD DATASET

# Dataset Columns:
# target, ids, date, flag, user, text

columns = [
    'target',
    'ids',
    'date',
    'flag',
    'user',
    'text'
]

df = pd.read_csv(
    '/content/twitterdataset.csv',
    encoding='latin-1',
    names=columns
)

print(df.head())

ParserError: Error tokenizing data. C error: EOF inside string starting at row 954553

In [ ]:
# Select Required Columns

df = df[['target', 'text']]

# Convert labels:
# 0 = Negative
# 4 = Positive

df['target'] = df['target'].replace(4, 1)


# Reduce dataset size

df = df.sample(50000, random_state=42)

print("\nDataset Shape:", df.shape)

In [ ]:
# TEXT PREPROCESSING


stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+", "", text)

    # Remove mentions
    text = re.sub(r"@\w+", "", text)

    # Remove hashtags
    text = re.sub(r"#\w+", "", text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()

    cleaned_words = []

    for word in words:

        if word not in stop_words:

            stemmed_word = stemmer.stem(word)

            cleaned_words.append(stemmed_word)

    return " ".join(cleaned_words)


df['clean_text'] = df['text'].apply(clean_text)

print("\nCleaned Text Example:")
print(df['clean_text'].iloc[0])


In [ ]:
# WORD CLOUD VISUALIZATION

all_words = " ".join(df['clean_text'])

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(all_words)

plt.figure(figsize=(12,6))

plt.imshow(wordcloud)

plt.axis('off')

plt.title("Most Common Words in Tweets")

plt.show()


In [ ]:
# TRAIN TEST SPLIT

X = df['clean_text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# FEATURE EXTRACTION

# TF-IDF Vectorization

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)


# MODEL 1 - LOGISTIC REGRESSION

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_test_tfidf)


# MODEL 2 - NAIVE BAYES

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_pred = nb_model.predict(X_test_tfidf)


# MODEL 3 - SVM

svm_model = LinearSVC()

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)


In [ ]:
# EVALUATION FUNCTION


def evaluate_model(y_test, predictions, model_name):

    accuracy = accuracy_score(y_test, predictions)

    print("\n===================================")
    print(f"{model_name} Results")
    print("===================================")

    print("Accuracy:", accuracy)

    print("\nClassification Report:\n")

    print(classification_report(y_test, predictions))

    # -----------------------------
    # Confusion Matrix
    # -----------------------------
    cm = confusion_matrix(y_test, predictions)

    plt.figure(figsize=(5,4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(f"{model_name} Confusion Matrix")

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.show()

    return accuracy

In [ ]:
# MODEL EVALUATION


lr_acc = evaluate_model(
    y_test,
    lr_pred,
    "Logistic Regression"
)

nb_acc = evaluate_model(
    y_test,
    nb_pred,
    "Naive Bayes"
)

svm_acc = evaluate_model(
    y_test,
    svm_pred,
    "SVM"
)




In [ ]:
# FINAL COMPARISON


results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Naive Bayes',
        'SVM'
    ],
    'Accuracy': [
        lr_acc,
        nb_acc,
        svm_acc
    ]
})

print("\n===================================")
print("FINAL MODEL COMPARISON")
print("===================================")

print(results)

In [ ]:
# COMPARISON GRAPH

plt.figure(figsize=(8,5))

sns.barplot(
    x='Model',
    y='Accuracy',
    data=results
)

plt.title("Model Accuracy Comparison")

plt.show()